In [ ]:
# Copyright (C) 2026 Analog Devices, Inc.
#
# SPDX short identifier: ADIBSD

from adi.ad5686 import ad5686

nanodac = ad5686("ip:192.168.101.2", device_name="ad5686r")

#### Channel Voltage

In [ ]:
nanodac.channel[0].voltage = 2.5
nanodac.channel[1].voltage = 2
nanodac.channel[2].voltage = 1.5
nanodac.channel[3].voltage = 1

In [ ]:
for ch in nanodac.channel:
    print(f"Channel {ch.name}: {ch.voltage} V ({ch.raw})")

Channel voltage0: 2.49996181002 V (65535)
Channel voltage1: 1.999969448016 V (52428)
Channel voltage2: 1.499977086012 V (39321)
Channel voltage3: 0.999984724008 V (26214)


#### Enable 2x Gain

In [ ]:
nanodac.set_gain(ad5686.gain.DOUBLE)

for ch in nanodac.channel:
    print(f"Channel {ch.name}: {ch.voltage} V ({ch.raw})")

Channel voltage0: 4.999923685575 V (65535)
Channel voltage1: 3.99993894846 V (52428)
Channel voltage2: 2.999954211345 V (39321)
Channel voltage3: 1.99996947423 V (26214)


In [ ]:
nanodac.set_gain(ad5686.gain.NORMAL)

#### Buffered Output

In [ ]:
from adi.trigger import hrtimer_trig
import numpy as np

trig0 = hrtimer_trig(uri=nanodac._ctx.attrs["uri"])
trig0.sampling_frequency = 1000

nanodac.set_tx_trigger(trig0)

In [ ]:
# processed (scaled) values are in mV
voltage0_mV = np.linspace(0, 2500, num=30)
voltage1_mV = np.linspace(2500, 0, num=30)

nanodac.tx_enabled_channels = [0, 1]
nanodac.tx_input_type = "SI"
nanodac.tx([voltage0_mV, voltage1_mV])

In [ ]:
# Plot of Expected Output:
import pandas as pd
import plotly.express as px

num_samples = len(voltage0_mV)
time = np.arange(num_samples) / trig0.sampling_frequency

fig = px.line(
    pd.DataFrame({
        'Time (s)': time,
        'Voltage 0': voltage0_mV,
        'Voltage 1': voltage1_mV
    }),
    x='Time (s)',
    y=['Voltage 0', 'Voltage 1']
)
fig.update_traces(line_shape="hv")
fig.show()